# Mini-Project 1: Where Does GPT-2 Small "Know" Country Capitals?

**Timebox:** 1–2 days (throwaway mini-project — aim for fast iteration & real plots, not perfection).

**Core Question:** When GPT-2 Small completes `"The capital of France is" -> " Paris"`, at which layers does the answer emerge, and do **Attention heads** or **MLPs** write it to the residual stream?

**TransformerLens primitives you'll practice:** `model.to_single_token`, `model.to_str_tokens`, `utils.test_prompt`, `model.run_with_cache`, `cache.accumulated_resid`, `cache.decompose_resid`, `cache.logit_attrs`.

In [ ]:
# Works in both Google Colab and local Jupyter (only installs on Colab)
import sys
if "google.colab" in sys.modules:
    %pip install -q transformer-lens einops plotly

import torch
import einops
import plotly.express as px
import plotly.graph_objects as go
from transformer_lens import HookedTransformer, utils

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = HookedTransformer.from_pretrained("gpt2-small", device=device)

## Step 1: Build & Inspect Your Dataset (Raw Data First!)

**TODO:**
1. Create a list of 25–30 `(country, capital)` pairs.
2. Check tokenization: filter to pairs where `" " + capital` is a **single token** in GPT-2 (`model.to_tokens` / `model.to_single_token`).
3. Run `utils.test_prompt` on a couple of examples, then check accuracy across all prompts of the form `f"The capital of {country} is"`.
4. Keep the prompts GPT-2 gets right (and note which ones it gets wrong — surprising failures are worth inspecting!).

In [ ]:
# TODO: Define your (country, capital) pairs and check tokenization + accuracy
pairs = [
    ("France", "Paris"),
    ("Germany", "Berlin"),
    ("Japan", "Tokyo"),
    # Add ~25 more...
]

# Hint: utils.test_prompt("The capital of France is", " Paris", model)


## Step 2: Logit Lens Across Layers

**TODO:**
1. Run `logits, cache = model.run_with_cache(prompts)` (or loop over prompts if they have different token lengths, grabbing the **final token position**).
2. Extract the accumulated residual stream after each layer (`cache.accumulated_resid(layer=-1, incl_mid=True, pos_slice=-1, return_labels=True)`).
3. Apply `cache.apply_ln_to_stack(...)` and project onto the unembedding direction for the correct capital (or logit difference: `correct_capital - wrong_capital`).
4. Plot the mean (and individual curves!) across layers. At which layer does the capital suddenly jump up?

In [ ]:
# TODO: Implement Logit Lens across layers and plot


## Step 3: Direct Logit Attribution (Attention vs. MLP)

**TODO:**
1. Decompose the final residual stream into per-layer components: `embed`, `pos_embed`, `0_attn_out`, `0_mlp_out`, ..., `11_mlp_out` using `cache.decompose_resid(layer=-1, pos_slice=-1, return_labels=True)`.
2. Compute each component's direct contribution to the correct capital logit (or logit diff) using `cache.logit_attrs(...)` or manual `einops.einsum` with `model.W_U`.
3. Plot a bar chart of average contribution per component. Are **MLPs** or **Attention heads** doing the heavy lifting?
4. *(Stretch)* If attention matters at a specific layer, decompose down to individual attention heads (`cache.stack_head_results`).

In [ ]:
# TODO: Implement Direct Logit Attribution (Attention vs MLP) and plot


## Step 4: Red-Team Your Finding!

Before believing your plot, try to **break** it:
1. **Rephrase the prompt:** Test at least 2 different templates:
   - `f"{country}'s capital city is"`
   - `f"In {country}, the capital is"`
   - Few-shot: `"France: Paris\nGermany: Berlin\nJapan:"`
2. Do the **same layers and MLPs/heads** dominate, or did your original plot mostly reflect the exact wording `"The capital of X is"`?
3. **Control / Baseline:** What if you measure `logit(correct_capital)` vs. `logit(correct_capital) - mean_logit(other_capitals)`? Does a layer just boost *all* city names instead of the *specific* capital?
4. Summarize what held up and what broke in `notes.md`.

In [ ]:
# TODO: Run rephrasing & city-baseline red-team checks
